# 七品特征与七日标签

版本seven-daily-notebook-1.0。把已有清洗数据整理成“过去输入＋七日真实答案”，本页用于核对样本，不训练模型、不请求市场。

直接运行全部查看已经构建的结果。最后七天的未来标签待成熟属于正常情况。字段和更新步骤见[说明](../../docs/data/SEVEN_DAILY_SAMPLES.md)。

In [ ]:
from pathlib import Path
from contextlib import closing
import sys, json, sqlite3, html
sys.dont_write_bytecode = True
from IPython.display import display, Markdown, HTML
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "scripts/build_seven_samples.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("请先在VS Code打开 F:/Codex/CS_YouOweMe 项目文件夹。")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from scripts.research.data import seven_samples as samples

def show_table(records, fields):
    if not records:
        print("没有可展示的记录。")
        return
    def value(v):
        return "缺失" if v is None else str(v)
    markup = "<table><thead><tr>" + "".join("<th>"+html.escape(k)+"</th>" for k in fields) + "</tr></thead><tbody>"
    markup += "".join("<tr>" + "".join("<td>"+html.escape(value(r.get(k)))+"</td>" for k in fields) + "</tr>" for r in records)
    display(HTML(markup + "</tbody></table>"))
print("项目：", PROJECT)

## 1. 读取现有版本

`RUN_BUILD=False`保持只读查看。只有新建版本时才改True；先明确选择新的清洗ID，已有版本不会覆盖。

In [ ]:
RUN_BUILD = False
CLEAN_ID = "seven_clean_20261019_v1"
DATASET_ID = None  # 查看latest；新建时None自动生成独立ID，或填未使用过的名字
RESULT = samples.build_samples(CLEAN_ID, DATASET_ID) if RUN_BUILD else samples.status(DATASET_ID)
OUTPUT = Path(RESULT["output_dir"]) if RESULT.get("output_dir") else None
if OUTPUT is None:
    print("尚无本地七品样本。先准备清洗版，再显式将RUN_BUILD改True构建。")
else:
    print("状态：", RESULT.get("status"))
    print("父清洗版：", RESULT.get("parent_clean_id"))
    print("数据截止：", RESULT.get("source_cutoff_beijing"))
    print("样本库：", OUTPUT / RESULT["database"])
    print("特征表：", OUTPUT / "features.csv")
    print("标签表：", OUTPUT / "labels.csv")
    print("完整报告：", OUTPUT / RESULT["report"])

## 2. 看七个商品的样本数量

严格输入与容缺输入分别统计；七日点价标签与完整路径标签也分别统计。质量合格、标签可用、时间隔离和正式训练批准是不同的事。

In [ ]:
if OUTPUT is not None:
    display(Markdown((OUTPUT / RESULT["report"]).read_text(encoding="utf-8")))

## 3. 查看最近样本的真实状态

下面最多显示最近两天的14行。未来尚未走完七天时，标签缺失和pending正常；不能补造未来答案。

In [ ]:
RECENT = []
if OUTPUT is not None:
    database = (OUTPUT / RESULT["database"]).resolve()
    with closing(sqlite3.connect(database.as_uri() + "?mode=ro", uri=True)) as db:
        db.row_factory = sqlite3.Row
        RECENT = [dict(r) for r in db.execute("SELECT * FROM samples ORDER BY decision_time DESC, market_hash_name LIMIT 14")]
    if RECENT:
        keys = list(RECENT[0])
        preferred = ["market_hash_name", "decision_at_beijing", "split", "x_reference_close", "x_return_7d"]
        states = [k for k in keys if k.endswith("_status") or "eligible" in k]
        fields = [k for k in preferred if k in keys] + states[:8]
        show_table(RECENT, fields)

## 4. 输入和答案分开看

真正的模型输入仅限下列冻结`FEATURE_FIELDS`。未来标签、未来可用状态和样本准入列用于核验，不作为预测输入。数值缺失仍显示“缺失”，没有改成0。

In [ ]:
if OUTPUT is not None and RECENT:
    current = RECENT[0]
    print("示例时点：", current.get("decision_at_beijing"), current["market_hash_name"])
    show_table([{"特征": k, "数值": current.get(k)} for k in samples.FEATURE_FIELDS], ["特征", "数值"])
    print("输入特征数：", len(samples.FEATURE_FIELDS))
    print("没有在本页拟合、预测或选择模型。")

## 下一步怎么做

先核对本页确实列出7个商品、样本截止时间、可用样本和待成熟标签。下一阶段建立适配这版样本的七品持平基准与简单模型对照，继续用新前向数据检验。

需要新数据时依次：采集Notebook显式daily有限更新 → 清洗Notebook新建清洗版 → 本页选择新`CLEAN_ID`并显式构建新样本。旧版保留；旧次时代预测Notebook仍显示其原有结果。